In [1]:
import os
import cv2
import numpy as np
import imgaug as ia
from imgaug import augmenters as iaa

def oversampling(path, max_images, disp):
    """
    Perform oversampling on the images in the specified path.

    Args:
    - path (str): Path to the directory containing the images.
    - max_images (int): Maximum number of images to generate.

    Returns:
    - list: List of oversampled images.
    """
    # Define the augmentation sequence
    aug_seq = iaa.Sequential([
        iaa.Fliplr(0.5),  # horizontal flip
        iaa.Flipud(0.5),  # vertical flip
        iaa.Affine(rotate=(-3, 3)),  # rotate by -3 to 3 degrees
        iaa.Add((-10, 10), per_channel=0.5),  # add noise
    ])
    
    # Load the images
    images = []
    for file in os.listdir(path):
        if file.endswith('.png') or file.endswith('.jpg'):
            img = cv2.imread(os.path.join(path, file))
            img = cv2.resize(img, img_size)
            images.append(img)
    
    # Perform oversampling until reaching max_images
    imgCnt = 0
    oversampled_images = []
    while len(oversampled_images) < max_images:
        for img in images:
            if len(oversampled_images) >= max_images:
                break
            aug_img = aug_seq.augment_image(img)
            oversampled_images.append(aug_img)
            imgCnt += 1
            if imgCnt == 20 and disp == 1:
                plt.imshow(aug_img)  # Display grayscale image
                plt.axis('off')
                plt.show()
                imgCnt = 0
    
    return oversampled_images

# Define the image size
img_size = (224, 224)

# Define the data directories
xs_path = '/kaggle/input/xoblank/X'
os_path = '/kaggle/input/xoblank/O'
blank_path = '/kaggle/input/xoblank/Blank'

max_images_to_generate = 100
disp = 0
oversampled_x_images = oversampling(xs_path, max_images_to_generate, disp)
print("Number of oversampled X class images:", len(oversampled_x_images))

oversampled_o_images = oversampling(os_path, max_images_to_generate,disp)
print("Number of oversampled O class images:", len(oversampled_o_images))

oversampled_blank_images = oversampling(blank_path, max_images_to_generate, disp)
print("Number of oversampled Blank class images:", len(oversampled_blank_images))


Number of oversampled X class images: 100
Number of oversampled O class images: 100
Number of oversampled Blank class images: 100


In [ ]:
images = np.concatenate((oversampled_blank_images,
                         oversampled_x_images,
                         oversampled_o_images))

# Blank -> 0  # X -> 1 # O -> 2
labels =  [0] * len(oversampled_blank_images) \
        + [1] * len(oversampled_x_images) \
        + [2] * len(oversampled_o_images)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_eval, y_train, y_eval = train_test_split(images, labels, test_size=0.1, random_state=42)

print(len(y_eval))

In [ ]:
import numpy as np

# Convert lists to NumPy arrays
y_train = np.array(y_train)
y_eval = np.array(y_eval)

# Reshape the arrays
X_train = X_train.reshape(-1, 224, 224, 1)
X_eval = X_eval.reshape(-1, 224, 224, 1)

# Normalize the images
X_train = X_train / 255.0
X_eval = X_eval / 255.0

from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)
y_train = encoder.fit_transform(y_train.reshape(-1, 1))
y_eval = encoder.transform(y_eval.reshape(-1, 1))

In [ ]:
# Example using Keras
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input

input_shape = (224, 224, 1)

model = Sequential()
model.add(Input(input_shape))
model.add(Conv2D(32, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(3, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()


In [ ]:
history = model.fit(X_train, y_train,
          epochs=10, batch_size=32,
          validation_data=(X_eval, y_eval),
          shuffle=True
         )

In [ ]:
loss, accuracy = model.evaluate(X_eval, y_eval)
print(f'Validation accuracy: {accuracy:.2f}')

In [ ]:
from sklearn.metrics import classification_report

X_test = X_eval
y_test = y_eval

# Evaluate the model
evalTest = model.evaluate(X_test, y_test)

# Get predictions
y_pred = model.predict(X_test)
y_pred_labels = np.argmax(y_pred, axis=1)

# Print classification report
#print(classification_report(y_test, y_pred_labels))



print("Test Accuracy:", evalTest)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train'], loc='upper left')
plt.show()


In [ ]:
c+=1
expModel = f"model_v.{c}_{history.history['accuracy'][-1]:.3f}_{history.history['loss'][-1]:.3f}.h5"
model.save(expModel)

In [ ]:
print(expModel)

In [ ]:

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array



prv = '/kaggle/working/model_v.21_1.000_3.449.h5'
prvModel = load_model(prv)


loaded_model = load_model(expModel)

imgSize = (224, 224)

import matplotlib.pyplot as plt

def test_model_on_folder_or_list(model, prvModel,  path_or_list, num_files=10):
    """
    Test a Keras model on images in a folder or a list of filenames and print the predicted classes.

    Args:
    - model: Trained Keras model.
    - path_or_list: Path to the folder containing images or a list of filenames.
    - num_files: Number of files to randomly select if a folder path is provided (default is 20).

    Returns:
    - None
    """

    # If path_or_list is a folder path, list files, shuffle them, and select num_files
    if os.path.isdir(path_or_list):
        filenames = os.listdir(path_or_list)
        np.random.shuffle(filenames)  # Shuffle filenames to randomize selection
        selected_filenames = filenames[:num_files]
    else:
        selected_filenames = path_or_list
    score = 0
    # Iterate over each image file
    for filename in selected_filenames:
        if filename.endswith('.png') or filename.endswith('.jpg'):
            # Load and preprocess the image
            if os.path.isdir(path_or_list):
                img = load_img(os.path.join(path_or_list, filename), target_size=imgSize, color_mode='grayscale')
            else:
                img = load_img(filename, target_size=imgSize, color_mode='grayscale')
            img_array = img_to_array(img)
            img_array = img_array.reshape((1,) + img_array.shape)  # Add batch dimension
            img_array = img_array / 255.0  # Normalize pixel values

            plt.imshow(img_array[0], cmap='gray')  # Display grayscale image
            plt.axis('off')
            plt.show()
            
            # Predict the class of the image
            
            prediction = model.predict(img_array)
            prvPred = prvModel.predict(img_array)
            
            predicted_class = np.argmax(prediction)
            prvPredicted_class = np.argmax(prvPred)

            # Calculate confidence score for the predicted class
            confidence = prediction[0, predicted_class] * 100
            prvConfidence = prvPred[0, prvPredicted_class] * 100

            # Print the predicted class and confidence score as a percentage
            category = {0: 'Blank', 1: 'X', 2: 'O'}
            if confidence > prvConfidence:
                score += 1
            else:
                score += -1
            print(f"Class: {category[predicted_class]}, Conf: {confidence} , prv: {prvConfidence} , P(X, O, Blank): {prediction}, score = {score}")
    
# Example usage:
print(f'testing with model: {expModel} and prv: {prv}')
test_model_on_folder_or_list(loaded_model, prvModel, "/kaggle/input/xoblank/Blank")
test_model_on_folder_or_list(loaded_model, prvModel, "/kaggle/input/pngxos")





In [ ]:
import os
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
import time

imgSize = (224, 224)

def test_model_on_folders(model, folder_paths, pdf_filename, num_files_per_folder=50):
    """
    Test a Keras model on images in multiple folders and save the results in a PDF.

    Args:
    - model: Trained Keras model.
    - folder_paths: List of paths to folders containing images.
    - pdf_filename: Filename for the resulting PDF.
    - num_files_per_folder: Number of files to randomly select from each folder (default is 10).

    Returns:
    - None
    """

    with PdfPages(pdf_filename) as pdf:
        model_info = "Testing Model: model_v.22_1.000_0.003.pdf"
        plt.text(0.5, 0.5, model_info, ha='center', fontsize=18, fontweight='bold')
        plt.axis('off')
        pdf.savefig()
        plt.close()
        for _ in range(num_files_per_folder):
            folder_path = np.random.choice(folder_paths)  # Randomly choose a folder
            if not os.path.isdir(folder_path):
                print(f"Warning: {folder_path} is not a valid directory.")
                continue

            filenames = os.listdir(folder_path)
            np.random.shuffle(filenames)
            selected_filename = np.random.choice(filenames)  # Randomly choose a filename from the folder
            selected_filename = selected_filename.decode('utf-8')  # Decode bytes to string

            if selected_filename.endswith('.png') or selected_filename.endswith('.jpg'):
                img = load_img(os.path.join(folder_path, selected_filename), target_size=imgSize, color_mode='grayscale')
                img_array = img_to_array(img)
                img_array = img_array.reshape((1,) + img_array.shape)
                img_array = img_array / 255.0

                start_time = time.time()  # Start time
                prediction = model.predict(img_array)
                end_time = time.time()  # End time
                time_taken = end_time - start_time  # Time taken for evaluation in seconds
                time_taken *= 1000
                predicted_class = np.argmax(prediction)
                confidence = prediction[0, predicted_class] * 100

                category = {0: 'Blank', 1: 'X', 2: 'O'}
                titleFig = f"Class: $\mathbf{{{category[predicted_class]}}}$, Confidence: $\mathbf{{{confidence:.2f}\%}}$\nProbabilities (Blank, X, O): ({prediction[0][0]:.5f}, {prediction[0][1]:.5f}, {prediction[0][2]:.5f})"

                plt.imshow(img_array[0])  # Display grayscale image
                plt.axis('on')
                plt.title(titleFig)
                plt.text(0.5, -0.1, f"Time taken: {time_taken:.4f} ms", ha='center', transform=plt.gca().transAxes)
                pdf.savefig()
                plt.close()

# Example usage:
folder_paths = ["/kaggle/input/xs-and-os-dataset/Xs", "/kaggle/input/xs-and-os-dataset/Os"]
pdf_filename = f"test.pdf"
modelIn = load_model('/kaggle/working/model_v.21_1.000_3.449.h5')
test_model_on_folders(modelIn, folder_paths, pdf_filename)


In [ ]:
import os
import numpy as np
from matplotlib import pyplot as plt
from tensorflow.keras.models import load_model
from keras.preprocessing.image import img_to_array, load_img


model = load_model('pathToModel')

img = load_img('pathToPng', target_size= (224, 224), color_mode='grayscale')
img_array = img_to_array(img)
img_array = img_array.reshape((1,) + img_array.shape)
img_array = img_array / 255.0

prediction = model.predict(img_array)
predicted_class = np.argmax(prediction)

category = {0: 'Blank', 1: 'X', 2: 'O'}
titleFig = f"Class: {category[predicted_class]}, Prob (Blank, X, O): ({prediction[0][0]:.5f}, {prediction[0][1]:.5f}, {prediction[0][2]:.5f})"
plt.title(titleFig)
plt.imshow(img_array[0]) 
plt.axis('on')

In [ ]:
model.save('/kaggle/working/model_v.22_1.000_0.003.h5')

In [ ]:
import os
import cv2
import numpy as np
import random
from keras.preprocessing.image import img_to_array, load_img
from matplotlib import pyplot as plt

import imgaug as ia
from imgaug import augmenters as iaa


 


def oversampling_for_prediction(path, max_images):
    """
    Perform oversampling on the images in the specified path.

    Args:
    - path (str): Path to the directory containing the images.
    - max_images (int): Maximum number of images to generate.

    Returns:
    - list: List of oversampled images in a format suitable for prediction.
    """
    images = []
    for file in os.listdir(path):
        if file.endswith('.png') or file.endswith('.jpg'):
            img = cv2.imread(os.path.join(path, file))
            img = cv2.resize(img, (224, 224))  # Assuming target size is (224, 224)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)  # Convert RGB to grayscale
            img = np.expand_dims(img, axis=-1)  # Add channel dimension
            img_array = img_to_array(img)
            img_array = img_array / 255.0  # Normalize the image
            images.append(img_array)
            
    aug_seq = iaa.Sequential([
        iaa.Fliplr(0.5),  # horizontal flip
        iaa.Flipud(0.5),  # vertical flip
        iaa.Affine(rotate=(-45, 45)),  # rotate by -45 to 45 degrees
        #iaa.Add((-10, 10), per_channel=0.5),  # add noise
        iaa.Multiply((0.5, 1.5), per_channel=0.5),  # change brightness
        iaa.contrast.LinearContrast((0.5, 1.5), per_channel=0.5),  # change contrast
    ])
    
    imgCnt = 0
    oversampled_images = []
    while len(oversampled_images) < max_images:
        for img in images:
            if len(oversampled_images) >= max_images:
                break
            
            aug_img = aug_seq.augment_image(img)
            oversampled_images.append(aug_img)
            imgCnt += 1
            if imgCnt == 100:
                plt.imshow(aug_img)  # Display grayscale image
                plt.axis('off')
                plt.show()
                imgCnt = 0
    
    return oversampled_images

def select_random_images(images, num_images):
    """
    Select random images from the given list.

    Args:
    - images (list): List of images.
    - num_images (int): Number of images to select.

    Returns:
    - list: List of randomly selected images.
    """
    if len(images) <= num_images:
        return images
    else:
        return random.sample(images, num_images)

# Example usage
xs_path = '/kaggle/input/xoblank/X'
os_path = '/kaggle/input/xoblank/O'
blank_path = '/kaggle/input/xoblank/Blank'
max_images_to_generate = 200
num_random_images = 10

# Generate oversampled images for each class
oversampled_x_images = oversampling_for_prediction(xs_path, max_images_to_generate)
oversampled_o_images = oversampling_for_prediction(os_path, max_images_to_generate)
oversampled_blank_images = oversampling_for_prediction(blank_path, max_images_to_generate)

# Combine all oversampled images into one list
all_images = oversampled_x_images + oversampled_o_images + oversampled_blank_images


In [ ]:

# Select random images from the combined list
random_images = select_random_images(all_images, num_random_images)

# Now you can use these random images for prediction
for img_array in random_images:
    prediction = model.predict(np.expand_dims(img_array, axis=0))
    predicted_class = np.argmax(prediction)
    category = {0: 'Blank', 1: 'X', 2: 'O'}
    confidence = prediction[0, predicted_class] * 100
    # Display the image
    plt.imshow(img_array, interpolation='nearest')
    plt.show()
    print(f"Predicted Class: {category[predicted_class]}, conf = {confidence}")


In [ ]:
import os
import numpy as np
from matplotlib import pyplot as plt
from tensorflow.keras.models import load_model
from keras.preprocessing.image import img_to_array, load_img


model = load_model('pathToModel')

img = load_img('pathToPng', target_size= (224, 224), color_mode='grayscale')
img_array = img_to_array(img)
img_array = img_array.reshape((1,) + img_array.shape)
img_array = img_array / 255.0

prediction = model.predict(img_array)
predicted_class = np.argmax(prediction)

category = {0: 'Blank', 1: 'X', 2: 'O'}
titleFig = f"Class: {category[predicted_class]}, Prob (Blank, X, O): ({prediction[0][0]:.5f}, {prediction[0][1]:.5f}, {prediction[0][2]:.5f})"
plt.title(titleFig)
plt.imshow(img_array[0]) 
plt.axis('on')